In [17]:
import pandas as pd
import numpy as np
import time
import multiprocessing
from collections import Counter

# Clustering
from sklearn.cluster import KMeans, DBSCAN
import hdbscan

# NLP y Tópicos
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel

In [18]:
df = pd.read_parquet("RMHD_con_emociones.parquet")

X_refined = np.load("Embeddings Refinados/embeddings_bge_reducidos_umap.npy")

In [19]:
# ==============================================================================
# 1. FUNCIÓN: EXTRACCIÓN DE TÓPICOS (c-TF-IDF)
# ==============================================================================
def extract_cluster_topics(df, cluster_col, top_words=5):
    """
    Agrupa los textos por clúster y calcula c-TF-IDF para extraer las palabras clave.
    Ignora el clúster -1 (ruido en DBSCAN/HDBSCAN).
    """
    # Filtrar ruido
    df_valid = df[df[cluster_col] != -1].copy()
    
    if df_valid.empty:
        return {}
        
    # Agrupar textos por clúster (un solo gran documento por clúster)
    docs_per_cluster = df_valid.groupby(cluster_col)['text_clean'].apply(lambda x: " ".join(x.astype(str))).reset_index()
    
    # Vectorización
    vectorizer = CountVectorizer(stop_words='english', min_df=1)
    X = vectorizer.fit_transform(docs_per_cluster['text_clean'])
    
    # c-TF-IDF
    ctfidf_model = ClassTfidfTransformer()
    ctfidf = ctfidf_model.fit_transform(X)
    
    words = np.array(vectorizer.get_feature_names_out())
    cluster_topics = {}
    
    for i, cluster_id in enumerate(docs_per_cluster[cluster_col]):
        scores = ctfidf[i].toarray().flatten()
        top_indices = scores.argsort()[::-1][:top_words]
        cluster_topics[cluster_id] = words[top_indices].tolist()
        
    return cluster_topics


In [20]:
import random # <--- Asegúrate de importar esto arriba
import multiprocessing
import numpy as np
from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel

# ==============================================================================
# 2. FUNCIÓN: COHERENCIA TEMÁTICA (MUESTRA ESTRATIFICADA 20%)
# ==============================================================================
def compute_baseline_coherence(df, cluster_col, cluster_topics, sample_fraction=0.2):
    """
    Calcula la coherencia C_v tomando una muestra aleatoria (ej. 20%) 
    de los comentarios de cada clúster válido, optimizado para memoria.
    """
    sampled_texts = []
    
    # 1. Muestreo estratificado por clúster
    for cluster_id, topics in cluster_topics.items():
        if cluster_id == -1: continue
            
        cluster_docs = df[df[cluster_col] == cluster_id]['text_clean'].dropna().tolist()
        
        # Calcular tamaño de muestra con protección
        n_sample = max(2, int(len(cluster_docs) * sample_fraction))
        n_sample = min(len(cluster_docs), n_sample)
        
        # Muestrear usando 'random' nativo de Python para evitar MemoryError
        if n_sample > 0:
            sampled = random.sample(cluster_docs, n_sample)
            for text in sampled:
                sampled_texts.append(str(text).lower().split())
                
    if len(sampled_texts) < 2 or not cluster_topics:
        return np.nan

    # 2. Diccionario Gensim
    dictionary = Dictionary(sampled_texts)
    dictionary_words = set(dictionary.token2id.keys())
    
    # 3. Validar tópicos
    topics_valid = []
    for topic in cluster_topics.values():
        clean_topic = list(dict.fromkeys([str(w) for w in topic if w is not None]))
        topic_valid = [w for w in clean_topic if w in dictionary_words]
        if len(topic_valid) >= 2:
            topics_valid.append(topic_valid)
            
    if not topics_valid:
        return np.nan
        
    # 4. Calcular Coherencia
    n_cores = max(1, multiprocessing.cpu_count() - 1)
    cm = CoherenceModel(
        topics=topics_valid,
        texts=sampled_texts,
        dictionary=dictionary,
        coherence="c_v",
        processes=n_cores
    )
    
    return cm.get_coherence()

In [22]:
# ==============================================================================
# 3. PIPELINE DE EJECUCIÓN (K-Means, DBSCAN, HDBSCAN)
# ==============================================================================
# Asegúrate de cargar tus datos primero:
# df = pd.read_parquet("RMHD_con_emociones.parquet")
# X_refined = np.load("Embeddings Refinados/embeddings_bge_reducidos_umap.npy")

baseline_results = []

# Iterar sobre subreddits (puedes usar df_small para pruebas)
for subreddit in sorted(df["subreddit"].unique()):
    print(f"\n========== Ejecutando Baselines para: {subreddit} ==========")
    
    idx = df[df["subreddit"] == subreddit].index
    df_sub = df.loc[idx].copy().reset_index(drop=True)
    X_sub = X_refined[idx]
    
    if len(df_sub) < 50:
        print("   [!] Muy pocos datos. Saltando...")
        continue
        
    # Definir modelos a probar
    # Nota: Los parámetros de DBSCAN/HDBSCAN dependen fuertemente de tu UMAP
    models = {
        "K-Means (k=20)": KMeans(n_clusters=min(50, len(df_sub)//10), random_state=42, n_init='auto'),
        #"DBSCAN (eps=0.5, min_pts=15)": DBSCAN(eps=0.5, min_samples=15),
        #"HDBSCAN (min_size=15)": hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5)
    }
    
    for model_name, model in models.items():
        print(f"   -> Entrenando {model_name}...")
        t0 = time.time()
        
        # 1. Ajustar modelo y predecir
        if "K-Means" in model_name:
            labels = model.fit_predict(X_sub)
        else:
            labels = model.fit_predict(X_sub)
            
        df_sub['cluster'] = labels
        
        # 2. Calcular métricas estructurales
        # Identificar ruido (-1)
        noise_mask = (labels == -1)
        noise_pct = noise_mask.mean() * 100
        
        # Clústeres válidos
        valid_labels = labels[~noise_mask]
        n_clusters = len(set(valid_labels))
        
        if n_clusters == 0:
            print(f"      [!] {model_name} clasificó todo como ruido.")
            continue
            
        # 3. Extraer tópicos
        cluster_topics = extract_cluster_topics(df_sub, 'cluster', top_words=5)
        
        # 4. Calcular coherencia (20% de muestra)
        coherence = compute_baseline_coherence(
            df=df_sub, 
            cluster_col='cluster', 
            cluster_topics=cluster_topics, 
            sample_fraction=0.20
        )
        
        t1 = time.time()
        print(f"      Clústeres: {n_clusters} | Ruido: {noise_pct:.1f}% | Coherencia: {coherence:.4f} | Tiempo: {t1-t0:.2f}s")
        
        # Guardar resultados
        baseline_results.append({
            "subreddit": subreddit,
            "modelo": model_name,
            "n_clusters": n_clusters,
            "porcentaje_ruido": round(noise_pct, 2),
            "coherencia_cv_20pct": round(coherence, 4) if pd.notna(coherence) else None,
            "tiempo_segundos": round(t1-t0, 2)
        })

# Exportar resultados a un DataFrame
df_baseline = pd.DataFrame(baseline_results)
df_baseline.to_excel("metricas_baseline_kmeans.xlsx", index=False)
print("\n[+] Ejecución de baselines finalizada y exportada a Excel.")


========== Ejecutando Baselines para: Anxiety ==========
   -> Entrenando K-Means (k=20)...
      Clústeres: 50 | Ruido: 0.0% | Coherencia: 0.6301 | Tiempo: 62.79s

========== Ejecutando Baselines para: SuicideWatch ==========
   -> Entrenando K-Means (k=20)...
      Clústeres: 50 | Ruido: 0.0% | Coherencia: 0.5789 | Tiempo: 81.95s

========== Ejecutando Baselines para: depression ==========
   -> Entrenando K-Means (k=20)...
      Clústeres: 50 | Ruido: 0.0% | Coherencia: 0.5876 | Tiempo: 105.93s

========== Ejecutando Baselines para: lonely ==========
   -> Entrenando K-Means (k=20)...
      Clústeres: 50 | Ruido: 0.0% | Coherencia: 0.5645 | Tiempo: 38.62s

========== Ejecutando Baselines para: mentalhealth ==========
   -> Entrenando K-Means (k=20)...
      Clústeres: 50 | Ruido: 0.0% | Coherencia: 0.5586 | Tiempo: 72.59s

[+] Ejecución de baselines finalizada y exportada a Excel.
